# Team 5 – Wanddickencheck A2 (MVP 0.1)

**Projekt:** Automatisierte Qualitätsprüfung von Holzrahmenbauwänden  
**Modul:** Data Management & BIM  
**Regel:** A2 – `Qto_WallBaseQuantities.Width > 0 m` für `IfcWall` mit `CH_Lehre.Scope = Holzrahmen-MVP`.

Dieses Notebook wurde **für Team 5** auf Basis der Unterrichtsübungen „01 · Einstieg: Colab und ein IFC-Modell“ erstellt. Es prüft zwei offizielle **Lehrmodelle** unter **derselben Regel und Objektauswahl**:

- **Negativ:** `Demo_Holzrahmen_fehlerhaft.ifc` (erwartet: `Width = 0 m` → `durchgefallen`)
- **Positiv:** `Demo_Holzrahmen_korrigiert.ifc` (erwartet: `Width ≈ 0.1143 m` → `bestanden`)

Die tatsächlichen Resultate werden erst beim **Ausführen** berechnet. Das Notebook ist **keine** fertig geprüfte IDS-Datei. Es führt die Anforderung A2 als eigene Python-Prüflogik aus. A1 und A3 folgen später.

**Verarbeitung externer Daten:** Google Colab lädt die öffentlich bereitgestellten Lehrmodelle auf eine Google-Laufzeit. Vertrauliche Projektmodelle hier nur verwenden, wenn die Weitergabe ausdrücklich erlaubt ist.

## 1. Setup

Die folgenden Zellen **der Reihe nach** ausführen oder **Laufzeit → Alle ausführen** wählen. Voraussetzung: Google Colab mit Python und Internetzugang.

Für die Wiederholbarkeit wird IfcOpenShell auf eine bestimmte Version festgelegt. GitHub ist nur Quelle der Lehrmodelle; diese werden **nicht** in euer Team-Repository kopiert.

In [ ]:
#@title IfcOpenShell installieren (fixe Version)
%pip -q install ifcopenshell==0.9.0.post1 pandas

import ifcopenshell, sys, pandas as pd
print("Python:", sys.version.split()[0])
print("IfcOpenShell:", ifcopenshell.version)
print("pandas:", pd.__version__)

## 2. Lehrmodelle beziehen

Die Dateinamen und Download-Adressen stammen aus dem **Unterrichts-Notebook 01**. Beide Varianten der Holzrahmenwand werden lokal in dieser Colab-Laufzeit gespeichert. Das SHA-256-Prüfsummenprotokoll ermöglicht später den Vergleich der Eingangsdaten.

**Fixierter Quellenstand:** GitHub-Commit `b271ce09e369d73e248a3d11f7af1410d98f4eb7`. Damit ändert sich der Dateninhalt nicht, wenn der Kurs die Lehrmodelle später aktualisiert.

In [ ]:
#@title Beide Modellversionen herunterladen
from pathlib import Path
import hashlib
import urllib.request

QUELLE = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/b271ce09e369d73e248a3d11f7af1410d98f4eb7/modelle/"
DATEIEN = {
    "negativ": "Demo_Holzrahmen_fehlerhaft.ifc",
    "positiv": "Demo_Holzrahmen_korrigiert.ifc",
}
Path("testdaten").mkdir(exist_ok=True)
pfade = {}
for fall, name in DATEIEN.items():
    ziel = Path("testdaten") / name
    urllib.request.urlretrieve(QUELLE + name, ziel)
    pfade[fall] = ziel
    sha256 = hashlib.sha256(ziel.read_bytes()).hexdigest()
    print(f"{fall}: {name} | SHA256 {sha256} | {ziel.stat().st_size} Byte")

## 3. Prüfregel A2 festlegen

| Kriterium | Festlegung |
|---|---|
| **Applicability / Auswahl** | `IfcWall` inklusive Unterklassen, aber nur wenn `CH_Lehre.Scope = Holzrahmen-MVP` |
| **Requirement / Anforderung** | `Qto_WallBaseQuantities.Width > 0 m` |
| **Einheit** | Umrechnung anhand der im IFC-Modell definierten Längeneinheit |
| **Fehlt** | `fehlend` – nicht als Nullwert behandeln |
| **Wert ≤ 0** | `durchgefallen` |
| **Wert > 0** | `bestanden` |

**Wichtig:** Die **gleiche** Prüfregel wird auf beide IFC-Dateien angewendet. Kein Grenzwert wird zwischen den Tests verändert. Bei einem künftigen IFC2X3-Modell kann der Mengenpfad anders lauten (`BaseQuantities.Width`); diese Version prüft bewusst den Datenpfad des **IFC4-Lehrmodells**.

In [ ]:
#@title Check-Funktionen definieren
import math
from datetime import datetime, timezone
import ifcopenshell.util.element as element
import ifcopenshell.util.unit as units

REGEL_ID = "A2"
ENTITY = "IfcWall"                 # schliesst Unterklassen ein
SCOPE_SET = "CH_Lehre"
SCOPE_FELD = "Scope"
SCOPE_WERT = "Holzrahmen-MVP"
QTO_SET = "Qto_WallBaseQuantities"
QTO_FELD = "Width"
GRENZWERT_M = 0.0
REGELTEXT = "Qto_WallBaseQuantities.Width > 0 m"


def bewerte(width_m):
    """Ergebnis und verständlichen Grund zurückgeben (fehlend != 0)."""
    if width_m is None:
        return "fehlend", "Pflichtwert Width fehlt"
    if not math.isfinite(width_m):
        return "durchgefallen", "Width ist kein endlicher Zahlenwert"
    if width_m > GRENZWERT_M:
        return "bestanden", "Width ist grösser als 0 m"
    return "durchgefallen", "Width muss grösser als 0 m sein"

# Unabhängiger kleiner Logiktest
assert bewerte(None)[0] == "fehlend"
assert bewerte(0.0)[0] == "durchgefallen"
assert bewerte(0.1143)[0] == "bestanden"
print("Basis-Logiktest bestanden.")

In [ ]:
#@title Prüfungen ausführen (gleicher Scope, gleiche Regel)
def pruefe_datei(datei, fall):
    modell = ifcopenshell.open(str(datei))
    einheitenfaktor = units.calculate_unit_scale(modell)  # Länge in Modell-Einheit -> Meter
    laufzeit = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    zeilen = []

    for wand in modell.by_type(ENTITY):
        psets = element.get_psets(wand, psets_only=True)
        scope = psets.get(SCOPE_SET, {}).get(SCOPE_FELD)
        if scope != SCOPE_WERT:
            continue
        qsets = element.get_psets(wand, qtos_only=True)
        roh = qsets.get(QTO_SET, {}).get(QTO_FELD)
        try:
            breite_m = float(roh) * einheitenfaktor if roh is not None else None
        except (ValueError, TypeError):
            breite_m = math.nan
        status, grund = bewerte(breite_m)
        zeilen.append({
            "GlobalId": wand.GlobalId,
            "Klasse": wand.is_a(),
            "Name": wand.Name or "",
            "Datenpfad": f"{QTO_SET}.{QTO_FELD}",
            "Ist_Wert_m": (round(breite_m, 6) if breite_m is not None and math.isfinite(breite_m) else ""),
            "Soll_Regel": REGELTEXT,
            "Regel_ID": REGEL_ID,
            "Ergebnis": status,
            "Grund": grund,
            "Modell": datei.name,
            "Modell_SHA256": hashlib.sha256(datei.read_bytes()).hexdigest(),
            "IFC_Schema": modell.schema,
            "IfcOpenShell_Version": ifcopenshell.version,
            "Pruefzeit_UTC": laufzeit,
        })

    # Eine leere Auswahl DARF NICHT als "bestanden" gelten.
    if not zeilen:
        raise RuntimeError(f"Keine Wände im Scope '{SCOPE_WERT}' in {datei.name}; Modell/Scope prüfen!")
    daten = pd.DataFrame(zeilen)
    print(f"{fall}: {len(daten)} geprüft | " + ", ".join(
        f"{name}: {(daten['Ergebnis'] == name).sum()}"
        for name in ("bestanden", "durchgefallen", "fehlend")
    ))
    return daten

resultate = {fall: pruefe_datei(pfad, fall) for fall, pfad in pfade.items()}
for fall, tabelle in resultate.items():
    print("\n" + fall.upper())
    display(tabelle[["GlobalId", "Klasse", "Datenpfad", "Ist_Wert_m", "Soll_Regel", "Ergebnis", "Grund"]])

## 4. Erwartung gegen **tatsächlichen** Check abgleichen

Die Lehrmodelle sollen jeweils genau **eine** Wand mit identischer `GlobalId` im gewählten Scope enthalten. Der negative Fall muss fehlschlagen, der positive muss bestehen. Wenn eine dieser Annahmen nicht stimmt, stoppt die Prüfung und das Problem muss untersucht werden.

In [ ]:
#@title Validierung / kein blindes "grün"
negativ = resultate["negativ"]
positiv = resultate["positiv"]

assert len(negativ) == len(positiv) == 1, "Erwartete Objektanzahl stimmt nicht"
assert negativ["GlobalId"].tolist() == positiv["GlobalId"].tolist(), "GlobalIds unterschiedlich"
assert negativ["Ergebnis"].tolist() == ["durchgefallen"], "Negativtest nicht durchgefallen"
assert positiv["Ergebnis"].tolist() == ["bestanden"], "Positivtest nicht bestanden"
assert negativ["Ist_Wert_m"].iloc[0] == 0.0, "Negativwert nicht 0 m"
assert positiv["Ist_Wert_m"].iloc[0] > 0.0, "Positivwert nicht > 0 m"
print("VALIDIERT: Gleiche GlobalId + gleicher Scope + dieselbe Regel; negativer Fall fällt durch, positiver besteht.")

## 5. CSV-Ergebnisse exportieren

Die CSV-Dateien gehören später in euer persönliches Team-Repository unter `results/`. Sie werden hier mit **Semikolon** als Trennzeichen erstellt. `checks/` erhält **dieses Notebook**. Die Kursmodelle bleiben ausserhalb eures GitHub-Repositories, wenn die Freigabe zur Veröffentlichung nicht geklärt ist.

In [ ]:
#@title CSVs erzeugen und als ZIP herunterladen
from zipfile import ZipFile
ordner = Path("results")
ordner.mkdir(exist_ok=True)
exporte = []
for fall in ("negativ", "positiv"):
    ausgabe = ordner / f"Team_5_A2_Wanddicke_{fall}.csv"
    resultate[fall].to_csv(ausgabe, sep=";", index=False, encoding="utf-8-sig")
    exporte.append(ausgabe)
    print("Gespeichert:", ausgabe)

zip_pfad = Path("Team_5_A2_Ergebnisse.zip")
with ZipFile(zip_pfad, "w") as paket:
    for datei in exporte:
        paket.write(datei, arcname=datei.name)
print("ZIP bereit:", zip_pfad)
try:
    from google.colab import files
    files.download(str(zip_pfad))
except ImportError:
    print("Lokale Umgebung: ZIP unter obigem Pfad abrufen.")

## 6. Anleitung für die zentrale README.md (nach dem echten Lauf ergänzen)

1. Öffne `checks/Team_5_Wanddickencheck.ipynb` in Google Colab.
2. Wähle **Laufzeit → Alle ausführen** (ggf. Colab-Laufzeit starten).
3. Das Notebook installiert `ifcopenshell==0.9.0.post1` und lädt zwei **öffentliche Unterrichts-IFC-Modelle** herunter.
4. Es wählt alle `IfcWall` mit `CH_Lehre.Scope = Holzrahmen-MVP` aus und prüft **immer** `Qto_WallBaseQuantities.Width > 0 m`.
5. Es zeigt für beide Modelle die Anzahl geprüfter, bestandener, durchgefallener und fehlender Objekte.
6. Die beiden Fälle werden miteinander verglichen (gleiche `GlobalId`, gleiche Auswahl, gleiche Regel).
7. Die ZIP-Datei herunterladen; CSV-Dateien unter `results/` nach GitHub hochladen.
8. **Tatsächlichen** Laufzeitpunkt, Versionsangaben, Quell-URL und SHA-256-Werte (Ausgabe der Zellen) in der Abgabedokumentation festhalten.

**Grenzen:** Der Check kontrolliert ausschliesslich das Vorhandensein einer **positiven IFC-Wanddicke** im spezifizierten Datenpfad. Nicht geprüft werden die bautechnische Plausibilität der Dicke, die Vollständigkeit der Auswahl ausserhalb des gekennzeichneten Lehr-Scopes sowie die anderen Anforderungen A1 (Name) und A3 (Material). Das reale Arbeitsmodell von Team 5 (`IFC2X3`) ist nicht Gegenstand dieses **Lehrdaten-MVP**; dafür muss die IFC2X3-Auswahl und der dortige Datenpfad separat getestet werden.

**KI_USAGE:** Wenn dieses Notebook übernommen wird, die Unterstützung von ChatGPT bei der Erstellung und die eigenen Prüfschritte in `KI_USAGE.md` ergänzen.